# 11.1 Networking Fundamentals

**Prerequisites:** 07 Module and Packages, 08 File Handling (sockets behave a lot like files)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What actually happens when two programs talk over a network
- The four layers you need, and the two you can ignore for now
- IP addresses, with the `ipaddress` module
- **Ports** - well-known, registered and ephemeral
- DNS: turning a name into an address
- 🔴 **TCP vs UDP** - the single most important choice you make
- The shape of the socket API, before writing one
- **Byte order** and why `struct` exists
- 🔴 Sockets carry `bytes`, never `str`

---

## What a network actually is

Two programs, usually on different machines, sending sequences of bytes to each other. That is genuinely all of it. Everything else — protocols, ports, DNS — exists to answer three questions:

1. **Which machine?** → an IP address
2. **Which program on that machine?** → a port
3. **What do the bytes mean?** → a protocol (HTTP, SMTP, or one you invent)

### The postal analogy

| Post | Network |
|---|---|
| Street address | IP address — *which building* |
| Flat number | Port — *which door in that building* |
| Postal service | TCP/IP — *how it gets there* |
| Language in the letter | Protocol — *how to read it* |

The analogy holds surprisingly far. A letter to the right building with the wrong flat number is undeliverable — which is exactly `ConnectionRefusedError`: the machine answered, but nothing was listening on that port.

## The layers

Textbooks show seven OSI layers. In practice four matter, and you will work in the top one almost always.

```
   ┌─────────────────────────────────────────────────┐
   │ Application   HTTP, SMTP, SSH, your own protocol │  <- you are here
   ├─────────────────────────────────────────────────┤
   │ Transport     TCP / UDP        ports, delivery   │  <- and you choose here
   ├─────────────────────────────────────────────────┤
   │ Internet      IP               addressing, routing│
   ├─────────────────────────────────────────────────┤
   │ Link          Ethernet, Wi-Fi  the actual wire   │
   └─────────────────────────────────────────────────┘
```

Each layer wraps the one above it in its own header — like putting a letter in an envelope, in a sack, on a lorry. Your `b"hello"` gets a TCP header, then an IP header, then an Ethernet header, and the receiving machine unwraps them in reverse.

**Why this matters to you:** the socket API sits exactly on the line between Application and Transport. Everything below it is handled for you. Everything above it — including *where one message ends and the next begins* — is yours.

## IP addresses

An IPv4 address is **four bytes**, written as four numbers 0-255. IPv6 is sixteen bytes, written in hex groups. The stdlib `ipaddress` module handles both, and it is far better than string manipulation.

Some ranges have special meanings worth recognising on sight:

| Range | Meaning |
|---|---|
| `127.0.0.0/8` | **Loopback** - this machine. `127.0.0.1` is `localhost` |
| `10.0.0.0/8`, `172.16.0.0/12`, `192.168.0.0/16` | **Private** - home and office LANs |
| `0.0.0.0` | "all interfaces" when binding; not a real destination |
| `169.254.0.0/16` | Link-local - usually means DHCP failed |

> **`is_private` is broader than "on a LAN".** It means *not routable on the public internet*, so loopback and link-local addresses report `True` as well. The categories above overlap; they are not mutually exclusive.

In [ ]:
import ipaddress

for text in ["127.0.0.1", "192.168.1.10", "8.8.8.8", "169.254.3.4", "::1"]:
    addr = ipaddress.ip_address(text)
    tags = []
    if addr.is_loopback:
        tags.append("loopback")
    if addr.is_private:
        tags.append("private")
    if addr.is_global:
        tags.append("global")
    if addr.is_link_local:
        tags.append("link-local")
    print(f"  {text:<16} IPv{addr.version}  {', '.join(tags)}")

print("\n-- a network, not just an address --")
net = ipaddress.ip_network("192.168.1.0/24")
print("  network      :", net)
print("  netmask      :", net.netmask)
print("  usable hosts :", net.num_addresses - 2)
print("  first three  :", [str(h) for h in list(net.hosts())[:3]])
print("  contains .10 :", ipaddress.ip_address("192.168.1.10") in net)
print("  contains .5.1:", ipaddress.ip_address("192.168.5.1") in net)

print("\n-- an address really is just four bytes --")
packed = ipaddress.ip_address("192.168.1.10").packed
print("  packed:", packed, "=", list(packed))

## Ports

A port is a 16-bit number: **0 to 65535**. It identifies which program on a machine should receive the data.

| Range | Name | Notes |
|---|---|---|
| 0-1023 | **Well-known** | HTTP 80, HTTPS 443, SSH 22. Binding these needs admin rights on Unix |
| 1024-49151 | **Registered** | PostgreSQL 5432, MySQL 3306, Redis 6379 |
| 49152-65535 | **Ephemeral** | Handed out automatically for outgoing connections |

🔴 **Binding to port 0 means "give me any free port".** This is the single most useful trick for tests and examples: no collisions, no `Address already in use` when you re-run, no need to pick a number and hope. Every server in this folder does it.

In [ ]:
import socket

print("-- what the well-known names resolve to --")
for name in ["http", "https", "ssh", "smtp", "domain"]:
    try:
        print(f"  {name:<8} -> port {socket.getservbyname(name)}")
    except OSError:
        print(f"  {name:<8} -> not in this machine's services table")

print("\n-- port 0: the OS picks a free one --")
chosen = []
socks = []
for _ in range(3):
    s = socket.socket()
    s.bind(("127.0.0.1", 0))          # 0 = anything free
    chosen.append(s.getsockname()[1])
    socks.append(s)
print("  three sockets got:", chosen)
print("  all different    :", len(set(chosen)) == 3)
for s in socks:
    s.close()

print("\n-- what happens when nothing is listening --")
probe = socket.socket()
probe.settimeout(1.0)
try:
    probe.connect(("127.0.0.1", chosen[0]))   # a port we just closed
    print("  unexpectedly connected - something else grabbed that port")
except ConnectionRefusedError as exc:
    print("  ConnectionRefusedError:", exc.strerror)
    print("  ^ the machine answered and said 'nothing is listening there'.")
except (TimeoutError, socket.timeout):
    print("  timed out after 1s")
    print("  ^ no answer at all. Something - usually a firewall - dropped")
    print("    the packet instead of refusing it.")
finally:
    probe.close()

print()
print("Both outcomes are worth recognising, and they mean different things:")
print("  refused -> reached the machine, no program on that port")
print("  timeout -> never got an answer; firewall, wrong address, or host down")
print("Which one you get here depends on your firewall, so either is fine.")

## DNS - names to addresses

You type `example.com`; your machine needs `93.184.x.x`. DNS is the lookup, and it happens **before** any connection is made — a DNS failure and a connection failure are different problems with different fixes.

| Function | Gives you |
|---|---|
| `gethostbyname(name)` | one IPv4 address — legacy, but simple |
| `getaddrinfo(host, port)` | **all** results, IPv4 *and* IPv6 — what you should use |
| `gethostname()` | this machine's name |

The cell below needs the internet for the external lookup and says so if it is unavailable. `localhost` always resolves — it does not touch the network at all.

In [ ]:
import socket

print("this machine :", socket.gethostname())
print("localhost    :", socket.gethostbyname("localhost"), " <- no network needed")

print("\n-- an external lookup (needs the internet) --")
socket.setdefaulttimeout(3.0)
try:
    infos = socket.getaddrinfo("example.com", 443, proto=socket.IPPROTO_TCP)
    seen = []
    for family, _type, _proto, _canon, sockaddr in infos:
        label = "IPv6" if family == socket.AF_INET6 else "IPv4"
        if (label, sockaddr[0]) not in seen:
            seen.append((label, sockaddr[0]))
    for label, addr in seen:
        print(f"  {label}: {addr}")
    print("\n  getaddrinfo returns every route, which is why it is preferred")
    print("  over gethostbyname - that one only ever gives you IPv4.")
except OSError as exc:
    print("  no internet, or DNS is blocked:", exc)
    print("  Nothing else in this folder needs an external network -")
    print("  every server we write runs on 127.0.0.1.")
finally:
    socket.setdefaulttimeout(None)     # 🔴 always put this back

## 🔴 TCP vs UDP

The most consequential decision in network programming, and it is made in one argument to `socket()`.

| | **TCP** (`SOCK_STREAM`) | **UDP** (`SOCK_DGRAM`) |
|---|---|---|
| Connection | Yes — handshake first | No — just send |
| Delivery | Guaranteed, or you get an error | Best effort; may vanish silently |
| Order | Preserved | Not preserved |
| Duplicates | Removed | Possible |
| Message boundaries | 🔴 **None — it is a byte stream** | Preserved — one send, one receive |
| Overhead | Higher | Very low |
| Used by | HTTP, SSH, SMTP, databases | DNS, video calls, games, `syslog` |

### The line people get wrong

> TCP guarantees your **bytes** arrive in order. It guarantees **nothing** about where one *message* ends and the next begins.

Send `"AAA"`, `"BBB"`, `"CCC"` over TCP and the receiver may get `"AAABBBCCC"` in one read, or `"AA"` then `"ABBBC"` then `"CC"`. All are correct TCP behaviour. **11.2** demonstrates this happening and shows the fix.

UDP has the opposite trade: boundaries are preserved perfectly, and the datagram may simply never arrive — with no error.

**Choosing:** default to TCP. Choose UDP only when late data is worse than no data — live audio, video, telemetry — or when you are doing one tiny request/response and will handle retries yourself, which is why DNS uses it.

## The socket API, before we use it

A socket is an endpoint. The API is small, and the two sides use different halves of it.

```
   SERVER                              CLIENT
   ------                              ------
   socket()    create the endpoint     socket()
   bind()      claim an address
   listen()    start queuing arrivals
   accept()    wait for a client  <--- connect()   reach out
        |                                  |
   recv()/send()  <-------------->  send()/recv()
        |                                  |
   close()                            close()
```

`accept()` is the one that surprises people: it returns a **new** socket for that one client. The listening socket keeps listening. So a server juggling ten clients has eleven sockets.

> **Sockets behave like the file objects from 08.** They are opened, read, written and closed; they support `with`; and forgetting to close one leaks a resource. `recv()` returning `b""` means end-of-stream, exactly as an empty read does on a file.

In [ ]:
import socket

print("-- address families --")
print("  AF_INET   =", int(socket.AF_INET), " IPv4")
print("  AF_INET6  =", int(socket.AF_INET6), " IPv6")

print("\n-- socket types --")
print("  SOCK_STREAM =", int(socket.SOCK_STREAM), " TCP")
print("  SOCK_DGRAM  =", int(socket.SOCK_DGRAM), " UDP")

print("\n-- creating one commits to nothing --")
tcp = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
udp = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
print("  tcp:", tcp)
print("  udp:", udp)
print("  default timeout:", tcp.gettimeout(), " <- None means BLOCK FOREVER")

# 🔴 The most important habit in this folder.
tcp.settimeout(2.0)
print("  after settimeout:", tcp.gettimeout(), "seconds")

tcp.close()
udp.close()
print("\n  closed. A socket left open holds a file descriptor.")

## 🔴 The wire carries `bytes`, never `str`

`send("hello")` raises `TypeError`. This trips up everyone once, and it is not the library being awkward — the network genuinely moves bytes, and *which* bytes a piece of text becomes depends on the encoding.

```
    "café".encode("utf-8")    -> b'caf\xc3\xa9'   4 chars, 5 bytes
    "café".encode("latin-1")  -> b'caf\xe9'       4 chars, 4 bytes
```

Same text, different bytes. Both sides must agree, and **UTF-8 is the answer** unless a protocol says otherwise. This is the same lesson as text files in **8.1** — a network connection is just a file you did not open.

In [ ]:
import socket

s = socket.socket()
try:
    s.send("hello")            # str, not bytes
except TypeError as exc:
    print("TypeError:", exc)
finally:
    s.close()

print("\n-- the same text in three encodings --")
text = "café ☕"
for enc in ("utf-8", "utf-16", "latin-1"):
    try:
        raw = text.encode(enc)
        print(f"  {enc:<8} {len(raw):>3} bytes  {raw!r}")
    except UnicodeEncodeError as exc:
        print(f"  {enc:<8} cannot represent it: {exc.reason}")

print("\n-- decoding with the wrong one --")
raw = text.encode("utf-8")
print("  correct  :", raw.decode("utf-8"))
print("  latin-1  :", raw.decode("latin-1"), " <- mojibake, and NO error raised")
print("\n  Silent corruption is worse than a crash. Agree on UTF-8.")

## Byte order (endianness), and why `struct` exists

Text is not the only thing that needs agreement. Send the number **1** as four bytes and there are two reasonable answers:

```
    big-endian     00 00 00 01     most significant byte first
    little-endian  01 00 00 00     least significant byte first   <- x86, ARM
```

Networks standardised on **big-endian**, which is why it is called **network byte order** — and why almost every machine you own has to convert.

`struct` does the conversion, with a prefix character choosing the order:

| Prefix | Means |
|---|---|
| `!` | network (big-endian) — **use this** |
| `<` | little-endian |
| `>` | big-endian |
| `=` | native, native size |

Format characters: `B` unsigned byte, `H` unsigned 2 bytes, `I` unsigned 4 bytes, `Q` unsigned 8 bytes.

In [ ]:
import struct
import sys

print("this machine is", sys.byteorder + "-endian\n")

value = 1
print(f"the number {value} as four bytes:")
print("  network (!I):", struct.pack("!I", value).hex(" "))
print("  little  (<I):", struct.pack("<I", value).hex(" "))

print("\n-- 🔴 what reading it with the wrong order costs you --")
on_the_wire = struct.pack("!I", 1)
print("  sent as network order   :", struct.unpack("!I", on_the_wire)[0])
print("  read as little-endian   :", struct.unpack("<I", on_the_wire)[0])
print("  ^ 1 became 16777216. No error - just a wrong number.")

print("\n-- packing a small header, as a real protocol would --")
#  version(1) | msg_type(1) | length(4)
header = struct.pack("!BBI", 2, 7, 1024)
print("  packed  :", header.hex(" "), f"({len(header)} bytes)")
print("  unpacked:", struct.unpack("!BBI", header))
print("  calcsize:", struct.calcsize("!BBI"), "bytes")

print("\n  This exact idea frames messages over TCP in 11.2.")

## Where this is going

| Notebook | Builds |
|---|---|
| **11.2** | A real TCP client and server, and the framing problem |
| **11.3** | UDP, and what "unreliable" means in practice |
| **11.4** | Serving many clients: threads, `selectors`, `asyncio` |
| **11.5** | HTTP - by hand over a socket, then with `requests` |

Every server in this folder binds to **`127.0.0.1` on port 0**, runs on a **daemon thread**, and sets a **timeout on every blocking call**. Nothing reaches the internet, nothing needs admin rights, and no cell can hang.

---

## Common Mistakes & Pitfalls

1. 🔴 **Passing `str` to `send()`.** Sockets take `bytes`. Encode explicitly, and use UTF-8 unless a protocol dictates otherwise.
2. 🔴 **Leaving the default timeout of `None`.** That means block forever. A single unguarded `recv()` will hang a program with no error and no output.
3. 🔴 **Assuming TCP preserves message boundaries.** It does not. See **11.2**.
4. **Forgetting byte order for binary protocols.** Reading a big-endian 1 as little-endian gives 16777216 — a wrong number, not an exception.
5. **Using `gethostbyname` in new code.** It is IPv4-only. `getaddrinfo` gives you both families.
6. **Hardcoding a port in tests and examples.** Bind to port 0 and ask the socket which port it got.
7. **Confusing a DNS failure with a connection failure.** They happen at different moments and have different fixes.
8. **Calling `socket.setdefaulttimeout()` and not restoring it.** It is global and affects every socket created afterwards.

## Best Practices

- Set a timeout on every socket, immediately after creating it.
- Use `getaddrinfo` and let it choose IPv4 or IPv6.
- Encode and decode explicitly with UTF-8; never rely on a default.
- Use `!` in every `struct` format that crosses a network.
- Bind test servers to `127.0.0.1` — never `0.0.0.0`, which exposes them to your whole network.
- Bind to port 0 in tests, then read the real port from `getsockname()`.
- Close sockets with `with` or `try/finally`; they hold file descriptors.
- Default to TCP. Reach for UDP only when late data is worse than missing data.

## Practice Exercises

Try these before moving on.

1. Use `ipaddress` to decide whether `172.16.5.4` and `172.32.5.4` are private. Why do they differ, given both start with 172?
2. Write `describe(host)` that prints every address `getaddrinfo` returns for a host, labelled IPv4 or IPv6, and handles a lookup failure with a clear message.
3. Bind ten sockets to port 0 and collect the ports. Are they sequential? Does the answer change if you close each one before opening the next?
4. Pack a header of version, type and length with `!BBI`, then deliberately unpack it as `<BBI`. Which field survives unchanged, and why?
5. Encode `"naïve café"` as UTF-8 and decode it as latin-1. Then do the reverse. One raises and one silently corrupts — explain which and why.
6. 🔴 Create a socket, do not set a timeout, and predict what `connect()` to an unroutable address such as `10.255.255.1` does. Then run it with a timeout set, and compare how long each takes to give up.